# 03 - Neo4j Backend

Load the four example datasets into `Neo4jGraph` using `drm.exemples` loaders.

In [ ]:
import importlib.util

# Comprovació de presència del paquet
package_to_check = 'drm'
spec = importlib.util.find_spec(package_to_check)

if spec is None:
    print(f'⚠️ {package_to_check} no està instal·lat. Iniciant instal·lació...')
    %pip install -q --upgrade git+https://github.com/CVC-DAG/drm-tools.git
    print("✅ Instal·lació completada. L'estat del kernel PODRIA requerir un reinici.")
else:
    print(f'✅ {package_to_check} ja està present al sistema. Saltant instal·lació.')


In [ ]:
%pip install -q --upgrade git+https://github.com/CVC-DAG/drm-tools.git
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("drm.exemples") is None:
    repo_root = Path.cwd().resolve()
    while repo_root != repo_root.parent and not (repo_root / "setup.py").exists():
        repo_root = repo_root.parent
    if (repo_root / "setup.py").exists():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root)])
    else:
        raise ModuleNotFoundError("drm.exemples not found. Install from a local clone with `pip install -e <repo_root>`.")


In [ ]:
import os
from drm import Neo4jGraph
from drm.exemples import (
    load_bibliografia_openalex,
    load_got_characters,
    load_karate_club,
    load_movies_sample,
)

In [ ]:
def neo4j_config(default_target="DEV"):
    target = os.environ.get("NEO4J_TARGET", default_target).upper()
    pref = f"NEO4J_{target}_"
    return {
        "target": target,
        "url": os.environ.get(f"{pref}URL", os.environ.get("NEO4J_URL", "bolt://localhost:7687")),
        "user": os.environ.get(f"{pref}USER", os.environ.get("NEO4J_USER", "neo4j")),
        "password": os.environ.get(f"{pref}PASSWORD", os.environ.get("NEO4J_PASSWORD", "secret")),
        "database": os.environ.get(f"{pref}DATABASE", os.environ.get("NEO4J_DATABASE", "neo4j")),
    }


cfg = neo4j_config()
print("Using Neo4j target:", cfg["target"], cfg["url"], cfg["database"])

graph = Neo4jGraph(
    url=cfg["url"],
    user=cfg["user"],
    password=cfg["password"],
    database=cfg["database"],
)

In [ ]:
movies_stats = load_movies_sample(graph, limit=25)
print("Movies loaded:", movies_stats)

got_stats = load_got_characters(graph, limit=40)
print("GoT loaded:", got_stats)

# These same loaders also work with Neo4jGraph, so we can mirror the
# NetworkX tutorial and compare the resulting graph structure.
karate_stats = load_karate_club(graph)
print("Karate loaded:", karate_stats)

biblio_stats = load_bibliografia_openalex(graph, query="graph database", per_page=10)
print("Bibliography loaded:", biblio_stats)

print("Node count:", len(graph.get_nodes()))
print("Edge count:", len(graph.get_edges()))

In [ ]:
graph.close()